<center><font size=10>Restaurant Review Analysis Using LLM-Based Prompt Engineering</center></font>

<center><img src="https://machinelearninggeek.com/wp-content/uploads/2021/09/Analyzing-Sentiment-of-Restaurant-Reviews.png" width=800/></center>

# Overview



- A global food aggregator wants to analyze customer reviews at scale to understand service quality, food, and overall satisfaction.
- This project uses **prompt engineering with a Large Language Model (Mistral-7B-Instruct)** — without RAG — to classify overall sentiment, aspect-level sentiment (food, service, ambience), extract liked/disliked features, and generate personalized responses.

# Dataset



- **restaurant_id** – Unique identifier for each restaurant.
- **rating_review** – Customer-provided rating.
- **review_full** – Full text of the customer review.

### Installing Required Libraries

In [ ]:
!pip install -q transformers==4.53.2 \
                  accelerate==1.8.1 \
                  bitsandbytes==0.46.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


- Installs the Hugging Face libraries (`transformers`, `accelerate`, `bitsandbytes`) needed to load and run the Mistral model.
- Restart the runtime after installation before running the remaining cells.

### Importing Libraries and Loading the Dataset

In [ ]:
import pandas as pd
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/GenAi/restaurant_reviews.csv")

### Exploring the Dataset

In [ ]:
# checking the first five rows of the data
data.head()

,restaurant_ID,rating_review,review_full
0,FLV202,5,"Totally in love with the Auro of the place, re..."
1,SAV303,5,Kailash colony is brimming with small cafes no...
2,YUM789,5,Excellent taste and awesome decorum. Must visi...
3,TST101,5,I have visited at jw lough/restourant. There w...
4,EAT456,5,Had a great experience in the restaurant food ...


In [ ]:
data.shape

(20, 3)

In [ ]:
# checking for missing values
data.isnull().sum()

,0
restaurant_ID,0
rating_review,0
review_full,0


In [ ]:
# creating a copy of the data
df = data.copy()

- Dataset contains 20 reviews with restaurant ID, rating, and review text.
- No missing values in any column.

### Loading the Mistral-7B Model with 8-bit Quantization

In [ ]:
file_name = '/content/drive/MyDrive/GenAi/config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    HF_TOKEN = config.get("HF_TOKEN")


# Store API credentials in environment variables
os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
#import torch
#from transformers import AutoTokenizer, AutoModelForCausalLM

#model_id = "mistralai/Mistral-7B-Instruct-v0.1"

#tokenizer = AutoTokenizer.from_pretrained(model_id)

#model = AutoModelForCausalLM.from_pretrained(
    #model_id,
    #load_in_8bit=True,
    #torch_dtype=torch.float16,
    #device_map="auto"
#)

In [ ]:
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_8bit=True,  # Load the model with 8-bit quantization
    torch_dtype=torch.float16,         # Use 16-bit floats on GPU
    device_map="auto",                 # Automatically assign GPU or CPU
    token=HF_TOKEN
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (3 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

- Model loads successfully using 8-bit quantization and FP16 precision for efficient memory usage.
- Model layers are automatically distributed across available hardware (`device_map="auto"`).

### Testing the Model with a Sample Query

In [ ]:
# Define the prompt (question)
prompt = "### Question: What is the capital of France?\n### Answer:"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(**inputs)

# Decode and print the output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Question: What is the capital of France?
### Answer: Paris


- Model correctly answers the test query, confirming it is ready for use on the review data.

### Defining a Reusable Query Function

In [ ]:
def query_mistral(prompt, query):
    """
    Queries the Mistral model with a given prompt and query.

    Args:
        prompt (str): The prompt for the model.
        query (str): The query to be answered by the model.

    Returns:
        str: The model's response.
    """
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    attention_mask = (inputs != pad_token_id).long()

    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=300,  # Adjust as needed
        do_sample=True,
        temperature=0.7,     # Adjust as needed
        top_p=0.9,           # Adjust as needed
        pad_token_id=pad_token_id  # Prevents warning
    )

    # Decode and print the output, skipping the input tokens
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response


- Wraps the prompt and review text into a chat-formatted query and returns the model's response.
- Generation uses `temperature=0.7` and `top_p=0.9` for balanced, focused outputs, capped at `max_new_tokens=300`.

## Sentiment Analysis of Reviews

### Classifying Overall Review Sentiment

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into the following categories:
    - Positive
    - Negative
    - Neutral

    And return in only JSON format. No extra text and analysis
    {"Sentiment":"Positive"}
"""

In [ ]:
def classify_sentiment(prompt, query):
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI response: {e}")
        print(f"Raw OpenAI response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None



In [ ]:
# Apply the classification function to each row in the DataFrame
df['Sentiment'] = df['review_full'].apply(lambda x: classify_sentiment(instruction_1, x)['Sentiment'] if classify_sentiment(instruction_1, x) else None)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
Negative,8
Positive,6
Neutral,6


- Negative sentiment (8) slightly exceeds positive and neutral (6 each) — indicating mild overall dissatisfaction across restaurants.

### Analyzing Sentiment by Aspect (Food Quality, Service, Ambience)

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are an AI analyzing restaurant reviews. Classify the following aspects in the review and classify the sentiment of each aspect as "Positive", "Negative", or "Neutral":
    1. "Food Quality"
    2. "Service"
    3. "Ambience"

    Output the overall sentiment and sentiment for each category in a JSON format with the following keys:
    {
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }

    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) for the corresponding JSON key value.
    Only return the JSON, do not return any other information.
"""

In [ ]:
def classify_aspect_sentiment(prompt, query):
    """
    Classifies the sentiment of aspects of the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_2).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the sentiment classification for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_sentiment'] = df['review_full'].apply(lambda x: classify_aspect_sentiment(instruction_2, x))

# Normalize the JSON results into separate columns
aspect_sentiment_df = pd.json_normalize(df['aspect_sentiment'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_sentiment_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Not A...",Negative,Not Applicable,Positive
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Neutral
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable


In [ ]:
df['Food Quality'].value_counts()

,count
Food Quality,
Positive,7
Negative,7
Neutral,4
Not Applicable,2


In [ ]:
df['Service'].value_counts()

,count
Service,
Negative,10
Positive,7
Not Applicable,2
Neutral,1


In [ ]:
df['Ambience'].value_counts()

,count
Ambience,
Positive,9
Neutral,5
Not Applicable,4
Negative,2


- Food quality feedback is evenly split — 7 negative vs. 7 positive mentions.
- Service is the most criticized aspect — 10 negative vs. 8 positive mentions.
- Ambience receives the most positive feedback — 9 positive vs. only 2 negative mentions.

### Extracting Liked and Disliked Features by Aspect

In [ ]:
# defining the instructions for the model
instruction_3 = """

You are an AI model assigned to analyze restaurant reviews. Your task is to extract the **specific features** that the customer **liked or disliked**, categorized under the following aspects of the dining experience:

* Food Quality
* Service
* Ambience

Return the result in the following strict JSON format:

{
  "Food Quality Features": ["specific liked/disliked features"],
  "Service Features": ["specific liked/disliked features"],
  "Ambience Features": ["specific liked/disliked features"]
}

**Instructions:**

* Only list **concrete features** (e.g., “taste”, “temperature”, “presentation”, “waiting time”, “staff behavior”, “lighting”, “music volume”) that are mentioned positively or negatively in the review.
* Do **not** include generic phrases like “liked feature” or “disliked feature”.
* If a particular aspect has no feature mentioned, return an empty list for that aspect.
* Output **only the JSON**, with keys exactly as specified. Do not add any explanations or comments.

"""

In [ ]:
def classify_features(prompt, query):
    """
    Extracts liked/disliked features from the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_3).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the extracted features for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_features'] = df['review_full'].apply(lambda x: classify_features(instruction_3, x))

# Normalize the JSON results into separate columns
aspect_features_df = pd.json_normalize(df['aspect_features'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_features_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience,aspect_features,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive,{'Food Quality Features': ['pizza straight fro...,"[pizza straight from the oven, hummus and pita...","[mask on staff, good sanitisation]","[quaint and cute outdoor and indoor interior, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive,"{'Food Quality Features': ['Margherita pizza',...","[Margherita pizza, Hummus, Pita, Fussili Alfre...",[],"[peaceful, plants]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste'], 'Service ...",[taste],"[Subham Barnwal, great service]",[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': [], 'Service Feature...",[],"[Ms. Laxmi, handling client needs, specialty, ...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['food', 'taste', 'p...","[food, taste, presentation]","[staff, waiting time, helpfulness, professiona...","[hotel, front desk, kids, comfort]"
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['modern food', 'att...","[modern food, attentive service, good, taste, ...",[attentive service],"[stylish decor, lively music, good ambience]"
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Not A...",Negative,Not Applicable,Positive,{'Food Quality Features': ['Pulled Duck Salad'...,"[Pulled Duck Salad, Mulled Wine, Holiday Coffee]",[],"[Friendly atmosphere, Soft lighting, Cozy feel]"
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Neutral,"{'Food Quality Features': ['tasty', 'average']...","[tasty, average]","[slow, attentive staff]",[]
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive,"{'Food Quality Features': ['flavorful', 'lacke...","[flavorful, lacked distinctive taste]","[inconsistent, friendly]",[cozy atmosphere]
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': ['well-prepared', 'r...","[well-prepared, reasonable prices]","[efficient, attentive staff]",[]


- Extracts specific liked/disliked features (e.g., taste, staff behavior, waiting time, decor) for each aspect.
- Enables restaurant-specific insights beyond a single sentiment label.

### Generating Personalized Customer Responses

In [ ]:
# defining the instructions for the model
instruction_4 = """
You are an AI analyzing restaurant reviews. Your task is to generate a **polite and empathetic response** directly based on the sentiment of the review.

Follow this structure:

* Start with a thank you for their feedback.
* Then:

  1. If the review is positive, say you’re glad they enjoyed the experience and that it would be great to have them again.
  2. If the review is neutral, thank them and ask what the restaurant could have done better.
  3. If the review is negative, apologize for the inconvenience and mention that the team will look into the concerns raised.

Constraints:

* Do not start with “Dear Customer” or any greeting.
* Only output the final response. No sentiment label, explanation, or extra text.
"""

In [ ]:
def generate_customer_response(prompt, query):
    """
    Generates a customer response based on the review using the Mistral model.

    Args:
        prompt (str): The prompt for the model (instruction_4).
        query (str): The review text.

    Returns:
        str: The generated customer response.
    """
    response_text = query_mistral(prompt, query)
    return response_text

In [ ]:
# Apply the classification function to each row in the DataFrame
df['customer_response'] = df['review_full'].apply(lambda x: generate_customer_response(instruction_4, x))

In [32]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience,aspect_features,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive,{'Food Quality Features': ['pizza straight fro...,"[pizza straight from the oven, hummus and pita...","[mask on staff, good sanitisation]","[quaint and cute outdoor and indoor interior, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive,"{'Food Quality Features': ['Margherita pizza',...","[Margherita pizza, Hummus, Pita, Fussili Alfre...",[],"[peaceful, plants]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste'], 'Service ...",[taste],"[Subham Barnwal, great service]",[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': [], 'Service Feature...",[],"[Ms. Laxmi, handling client needs, specialty, ...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['food', 'taste', 'p...","[food, taste, presentation]","[staff, waiting time, helpfulness, professiona...","[hotel, front desk, kids, comfort]"
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['modern food', 'att...","[modern food, attentive service, good, taste, ...",[attentive service],"[stylish decor, lively music, good ambience]"
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Not A...",Negative,Not Applicable,Positive,{'Food Quality Features': ['Pulled Duck Salad'...,"[Pulled Duck Salad, Mulled Wine, Holiday Coffee]",[],"[Friendly atmosphere, Soft lighting, Cozy feel]"
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Neutral,"{'Food Quality Features': ['tasty', 'average']...","[tasty, average]","[slow, attentive staff]",[]
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive,"{'Food Quality Features': ['flavorful', 'lacke...","[flavorful, lacked distinctive taste]","[inconsistent, friendly]",[cozy atmosphere]
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': ['well-prepared', 'r...","[well-prepared, reasonable prices]","[efficient, attentive staff]",[]


- Generates a tailored, empathetic response for each review based on its sentiment and content.

## Key Takeaways

- Built an end-to-end LLM pipeline — overall sentiment, aspect-level sentiment, feature extraction, and automated responses — using prompt engineering alone.
- Overall sentiment: negative (8) slightly outweighs positive and neutral (6 each).
- Service is the most criticized aspect (10 negative); Ambience is the strongest positive driver (9 positive); Food quality is evenly split (7 vs. 7).
- Next steps: validate against manually labeled data and refine prompts/parameters to improve accuracy further.

# Conclusion


- This project built an end-to-end restaurant review analysis system using prompt engineering alone, with the Mistral-7B-Instruct model as the backbone — no fine-tuning or retrieval required.

- Overall sentiment: Across the 20 reviews analyzed, negative sentiment (8) slightly outweighed positive and neutral (6 each), pointing to mild net dissatisfaction across the sampled restaurants.
Aspect-level sentiment: Breaking sentiment down by category revealed where the dissatisfaction concentrates — service is the weakest aspect (10 negative vs. 8 positive), food quality is evenly split (7 vs. 7), and ambience is the clear bright spot (9 positive vs. only 2 negative).
- Feature extraction: Moving beyond sentiment labels, the model pulled out concrete liked/disliked details (e.g., taste, staff behavior, waiting time, decor), turning generic sentiment scores into specific, actionable feedback restaurants can act on.
- Automated responses: The same pipeline closed the loop by drafting a tailored, empathetic reply for every review based on its content and sentiment — something that could plug directly into a customer service workflow.

# Pipeline Diagram

<center><img src="https://i.postimg.cc/sxnRK8cS/Restaurant-Review-Pipeline-Diagram.jpg" width=800/></center>

# <font size=6 color='skyblue'>Thank You</font>
___